# EDA Silver zone

In [1]:
import os

from dotenv import load_dotenv
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

load_dotenv('../../../../.env_file/infra.env'), load_dotenv('../../../../.env_file/app.env')

(True, True)

In [2]:
spark = (
    SparkSession.builder
    .appName('ingest data to silver zone')
    # delta lake support
    # .config("spark.jars.packages", "io.delta:delta-spark_4.1_2.13:4.2.0")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    # read/write S3
    .config("spark.hadoop.fs.s3a.access.key", os.getenv('AWS_ACCESS_KEY_ID'))
    .config("spark.hadoop.fs.s3a.secret.key", os.getenv('AWS_SECRET_ACCESS_KEY'))
    # read postgres
    .config(
        'spark.jars',
        """
        ../../../../spark/jar/trino-jdbc-480.jar,
        ../../../../spark/jar/hadoop-aws-3.4.2.jar,
        ../../../../spark/jar/bundle-2.29.52.jar,
        ../../../../spark/jar/delta-spark_4.1_2.13-4.3.0.jar,
        ../../../../spark/jar/delta-kernel-api-4.3.0.jar,
        ../../../../spark/jar/delta-kernel-defaults-4.3.0.jar,
        ../../../../spark/jar/delta-storage-4.3.0.jar,
        """
    )
    # Hive metastore
    .config("spark.hadoop.hive.metastore.uris", "thrift://localhost:9084")
    .config("spark.sql.catalogImplementation", "hive")
    .enableHiveSupport()
    .getOrCreate()
)

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/09 20:40:31 WARN Utils: Your hostname, debian-nblong, resolves to a loopback address: 127.0.1.1; using 192.168.1.37 instead (on interface wlp0s20f3)
26/08/09 20:40:31 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/08/09 20:40:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [4]:
silver_table = 'silver.nyc_taxi'
df_silver = spark.read.table(silver_table)

In [4]:
def value_counts(df:DataFrame, col:str) -> DataFrame:
    return df.groupBy(col).count().orderBy('count')

In [5]:
for idx, col in enumerate(df_silver.columns):
    if df_silver.dtypes[idx][1] == 'string' and col not in ['trip_key', 'source_file']:
        print(col)
        value_counts(df_silver, col).show()
        print()

vendor_id


+--------------------+------+
|           vendor_id| count|
+--------------------+------+
|                 DDS| 32948|
|Creative Mobile T...|254864|
|  Curb Mobility, LLC|270573|
+--------------------+------+


store_and_fwd_flag


+------------------+------+
|store_and_fwd_flag| count|
+------------------+------+
|               Yes|  3593|
|                No|155782|
|              NULL|399010|
+------------------+------+


payment_type


+------------+------+
|payment_type| count|
+------------+------+
|     Dispute|   306|
|   No charge|  1696|
| Credit card|130483|
|        Cash|425900|
+------------+------+




In [8]:
df_silver.show()

+--------------------+-------------------+-------------------+---------------+------------------+------------+------------------+------------------+---------------+------------------+----------------+--------------+--------------+------------+-----------------+-----+-------+---------+---------------------+----------+------------+------------+--------------------+-----------+------------------+--------------------+-----------------------+-----------------------+----+-----+--------------------+-----------------------+-----------------------+
|           vendor_id|    pickup_datetime|   dropoff_datetime|passenger_count|     trip_distance|rate_code_id|store_and_fwd_flag|  pickup_longitude|pickup_latitude| dropoff_longitude|dropoff_latitude|pu_location_id|do_location_id|payment_type|      fare_amount|extra|mta_tax|surcharge|improvement_surcharge|tip_amount|tolls_amount|total_amount|congestion_surcharge|airport_fee|cbd_congestion_fee|         source_file|source_created_datetime|bronze_create

In [10]:
df_silver.describe().show()

+-------+--------------------+------------------+-----------------+------------------+------------------+------------------+-----------------+------------------+-----------------+------------------+-----------------+------------+------------------+-------------------+-------------------+-------------------+---------------------+------------------+-------------------+------------------+--------------------+-----------+------------------+--------------------+------------------+------------------+--------------------+
|summary|           vendor_id|   passenger_count|    trip_distance|      rate_code_id|store_and_fwd_flag|  pickup_longitude|  pickup_latitude| dropoff_longitude| dropoff_latitude|    pu_location_id|   do_location_id|payment_type|       fare_amount|              extra|            mta_tax|          surcharge|improvement_surcharge|        tip_amount|       tolls_amount|      total_amount|congestion_surcharge|airport_fee|cbd_congestion_fee|         source_file|              year

In [11]:
total_rows = df_silver.count()

null_counts_df = df_silver.select([
    F.sum(
        F.when(
            F.col(c).isNull() | (F.isnan(F.col(c)) if t in ("double", "float") else F.lit(False)),
            1
        ).otherwise(0)
    ).alias(c)
    for c, t in df_silver.dtypes
])

stack_expr = "stack({}, {}) as (column_name, null_count)".format(
    len(df_silver.columns),
    ", ".join([f"'{c}', `{c}`" for c in df_silver.columns])
)

null_stats_df = (
    null_counts_df
    .selectExpr(stack_expr)
    .withColumn("total_rows", F.lit(total_rows))
    .withColumn("null_ratio_pct", F.round(F.col("null_count") / F.col("total_rows") * 100, 4))
    .orderBy(F.desc("null_ratio_pct"))
)

null_stats_df.show(len(df_silver.columns), truncate=False)

+-----------------------+----------+----------+--------------+
|column_name            |null_count|total_rows|null_ratio_pct|
+-----------------------+----------+----------+--------------+
|congestion_surcharge   |4059640   |4059640   |100.0         |
|airport_fee            |4059640   |4059640   |100.0         |
|cbd_congestion_fee     |4059640   |4059640   |100.0         |
|pu_location_id         |3359884   |4059640   |82.7631       |
|do_location_id         |3359884   |4059640   |82.7631       |
|extra                  |3359884   |4059640   |82.7631       |
|improvement_surcharge  |3359884   |4059640   |82.7631       |
|store_and_fwd_flag     |2240261   |4059640   |55.1837       |
|rate_code_id           |1680569   |4059640   |41.397        |
|mta_tax                |1392903   |4059640   |34.311        |
|pickup_longitude       |699756    |4059640   |17.2369       |
|pickup_latitude        |699756    |4059640   |17.2369       |
|dropoff_longitude      |699758    |4059640   |17.2369 

- Loại bỏ những cột có quá nhiều giá trị NULL: `congestion_surcharge, airport_fee, `